# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [73]:
import os
print(os.listdir("/content"))

['.config', 'Flyrank-ml-internship.', 'sample_data']


In [74]:
%cd /content/Flyrank-ml-internship.

/content/Flyrank-ml-internship.


In [75]:
!ls

AGENTS.md    docs		     notebooks	       scripts	   work
CLAUDE.md    Flyrank-ml-internship.  outputs	       SETUP.md
data	     GUIDE.md		     README.md	       skills
DATA_USE.md  LICENSE		     requirements.txt  submission


In [76]:
!ls -lh data/raw/

total 6.5M
-rw-r--r-- 1 root root 6.5M Aug  4 13:44 content_refresh_anonymized.csv


In [77]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [78]:
df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [79]:
!git clone https://github.com/FatimaNdeem/Flyrank-ml-internship.

fatal: destination path 'Flyrank-ml-internship.' already exists and is not an empty directory.


#SECTION 1

###Check two signals first
Signal verdicts:

1. Freshness signal:
CONFIRMED — Older content tends to show weaker performance trends, making freshness a useful refresh indicator.

2. Trend signal:
CONFIRMED — Content with declining trend_direction has negative trend_pct values, confirming that trend decline is a meaningful signal.




In [80]:
# Signal 1: Freshness signal
freshness_check = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    avg_trend=("trend_pct", "mean")
).reset_index()

freshness_check

,freshness_tier,n,avg_trend
0,0-30,20480,0.784405
1,181+,174,-6.781203
2,31-90,175,-7.373054
3,91-180,9171,-15.683224


In [81]:
# Signal 2: Trend signal
trend_check = df.groupby("trend_direction").agg(
    n=("content_id", "count"),
    avg_trend=("trend_pct", "mean")
).reset_index()

trend_check

,trend_direction,n,avg_trend
0,down,16262,-58.113830
1,flat,1152,NaN
2,new,2236,NaN
3,stable,5962,-3.185944
4,up,4388,190.673997


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Content Refresh Opportunity

I will prioritize content that shows signs of decline but still has potential value.
The rule looks for content that has a negative trend, has not been updated recently,
and has enough search demand to justify improvement.

The rule uses three signals:
- trend_pct: identifies content losing performance.
- days_since_last_update: identifies stale content.
- search_volume: identifies content with remaining search opportunity.

Reason codes:

REFRESH_DECLINING_CONTENT:
Content has declining performance and is a candidate for updating.

KEEP_MONITORING:
Content does not show enough evidence for immediate action.

In [82]:
import pandas as pd

df["reason_code"] = "KEEP_MONITORING"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline score ranks content by estimated refresh opportunity.

Higher scores are given to content that:
- has a stronger negative trend,
- has not been updated recently,
- has higher search demand.

The output contains:
- content_id
- client_id
- action_score
- reason_code
- action_label

In [83]:
import pandas as pd
import numpy as np
import os


queue = df.copy()


# ----------------------------
# Create baseline signals
# ----------------------------

# Larger value = stronger decline
queue["decline_score"] = (
    -queue["trend_pct"]
).clip(lower=0)


# Older content = higher refresh opportunity
queue["freshness_score"] = queue["days_since_last_update"]


# Search demand opportunity
queue["search_score"] = queue["search_volume"]


# Handle missing values
queue["decline_score"] = queue["decline_score"].fillna(0)
queue["freshness_score"] = queue["freshness_score"].fillna(0)
queue["search_score"] = queue["search_score"].fillna(0)


# ----------------------------
# Baseline action score
# ----------------------------

queue["action_score"] = (
    0.5 * queue["decline_score"]
    +
    0.3 * queue["freshness_score"]
    +
    0.2 * queue["search_score"]
)


# ----------------------------
# Reason codes
# ----------------------------

queue["reason_code"] = "KEEP_MONITORING"


queue.loc[
    queue["action_score"] >= queue["action_score"].quantile(0.75),
    "reason_code"
] = "REFRESH_DECLINING_CONTENT"


# ----------------------------
# Action label
# ----------------------------

queue["action_label"] = "Refresh Content"


# Rank
queue = queue.sort_values(
    "action_score",
    ascending=False
)


baseline_queue = queue[
    [
        "content_id",
        "client_id",
        "action_score",
        "reason_code",
        "action_label"
    ]
]


# Save CSV

os.makedirs(
    "work/outputs",
    exist_ok=True
)


baseline_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


baseline_queue.head(10)

,content_id,client_id,action_score,reason_code,action_label
12140,content_ef99c4abd9ab,client_3fdba35f04,14831.20,REFRESH_DECLINING_CONTENT,Refresh Content
28282,content_454cc6654c6e,client_3fdba35f04,12157.60,REFRESH_DECLINING_CONTENT,Refresh Content
6972,content_bf67a444faef,client_3fdba35f04,12141.20,REFRESH_DECLINING_CONTENT,Refresh Content
18701,content_deb54e9e19cd,client_3fdba35f04,12132.45,REFRESH_DECLINING_CONTENT,Refresh Content
17907,content_5ec29ae79c60,client_3fdba35f04,12131.20,REFRESH_DECLINING_CONTENT,Refresh Content
8055,content_cd6760921db8,client_3fdba35f04,9944.80,REFRESH_DECLINING_CONTENT,Refresh Content
13502,content_f76ccf7a7834,client_19581e27de,9925.55,REFRESH_DECLINING_CONTENT,Refresh Content
22788,content_ee4630879d03,client_3fdba35f04,9920.65,REFRESH_DECLINING_CONTENT,Refresh Content
16005,content_83e3da1394ac,client_19581e27de,9906.60,REFRESH_DECLINING_CONTENT,Refresh Content
15923,content_84fe9d0a707a,client_3fdba35f04,8137.95,REFRESH_DECLINING_CONTENT,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review

The top-ranked items were selected because they had the highest refresh opportunity scores.

Each review includes:
- recommended action,
- reason code,
- confidence note,
- what could make the recommendation incorrect.

In [84]:
top20 = baseline_queue.head(20).copy()

reviews = []

for index, row in top20.iterrows():

    review = {
        "content_id": row["content_id"],
        "action": row["action_label"],
        "reason_code": row["reason_code"],
        "confidence_note":
            "Medium confidence because this is a rule-based score using observed signals, but it does not include external ranking factors.",
        "what_would_make_it_wrong":
            "The decline could be caused by seasonality, external events, or factors not captured in the dataset."
    }

    reviews.append(review)


top20_review = pd.DataFrame(reviews)

top20_review

,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_ef99c4abd9ab,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
1,content_454cc6654c6e,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
2,content_bf67a444faef,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
3,content_deb54e9e19cd,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
4,content_5ec29ae79c60,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
5,content_cd6760921db8,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
6,content_f76ccf7a7834,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
7,content_ee4630879d03,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
8,content_83e3da1394ac,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."
9,content_84fe9d0a707a,Refresh Content,REFRESH_DECLINING_CONTENT,Medium confidence because this is a rule-based...,"The decline could be caused by seasonality, ex..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

Some lower-ranked recommendations may be weak because the baseline rule only uses three signals:
- trend decline
- freshness (days since last update)
- search volume

Possible weak picks:
- Content with a low score but a temporary decline may not actually need a refresh.
- Older content may be correctly performing and should not always be updated.
- Low search volume pages may not provide enough opportunity even if they show decline.

Leakage check:

The baseline score only uses information available at decision time:
- trend_pct
- days_since_last_update
- search_volume

No future performance labels, future windows, product flags, or outcome columns were used.

In [85]:
# Display some weak picks from the bottom of the ranked queue

weak_picks = baseline_queue.tail(5)

weak_picks

,content_id,client_id,action_score,reason_code,action_label
11714,content_ef93c98775d5,client_98a3ab7c34,0.3,KEEP_MONITORING,Refresh Content
2428,content_68906b136014,client_98a3ab7c34,0.3,KEEP_MONITORING,Refresh Content
2636,content_0180ba8e643b,client_98a3ab7c34,0.3,KEEP_MONITORING,Refresh Content
2777,content_c53b8d4bce75,client_98a3ab7c34,0.3,KEEP_MONITORING,Refresh Content
12125,content_5142c2b724f1,client_98a3ab7c34,0.3,KEEP_MONITORING,Refresh Content


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.